In [1]:
!pip install fastapi uvicorn pyngrok transformers accelerate bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 102.8 MB/s eta 0:00:0000:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cud

In [2]:
import os
import torch
import gc

os.system("pkill -f uvicorn")

try:
    del model
    del base_model
    del finetuned_model
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()


import time
time.sleep(3)
print("Memory cleared")

Memory cleared


In [3]:
# from fastapi import FastAPI
# from fastapi.middleware.cors import CORSMiddleware
# from pydantic import BaseModel
# from typing import List
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# import uvicorn
# from pyngrok import ngrok
# import threading
# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import login

# user_secrets = UserSecretsClient()

# huggingface_token = user_secrets.get_secret("huggingface token1")
# ngrok_auth_token = user_secrets.get_secret("ngrok auth token")

# login(token = huggingface_token)

# app = FastAPI()

# app.add_middleware(
#     CORSMiddleware,
#     allow_origins=["*"],
#     allow_methods=["*"],
#     allow_headers=["*"],
# )

# print("Loading model")
# model_name = "simonjayl/simon-chatbot"

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
# )

# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     device_map="auto"
# )
# print("Model loaded")

# class Message(BaseModel):
#     role: str
#     content: str

# class ChatRequest(BaseModel):
#     message: str
#     history: List[Message] = []

# def build_prompt(message: str, history: List[Message]) -> str:
#     conversation = ""
#     for msg in history:
#         if msg.role == "user":
#             conversation += f"Them: {msg.content}\n"
#         else:
#             conversation += f"Simon: {msg.content}\n"
#     conversation += f"Them: {message}"
#     return f"[INST] Respond to this conversation as Simon would, match his texting style, tone and vocabulary exactly.\n\n{conversation} [/INST]"

# @app.post("/chat")
# async def chat(request: ChatRequest):
#     prompt = build_prompt(request.message, request.history)
#     inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=100,
#         temperature=0.7,
#         do_sample=True,
#         pad_token_id=tokenizer.eos_token_id
#     )
#     response = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     response = response.split("[/INST]")[-1].strip()
#     return {"response": response}

# @app.get("/health")
# async def health():
#     return {"status": "ok"}

# ngrok.set_auth_token(ngrok_auth_token)
# public_url = ngrok.connect(8000)
# print(f"\nPublic URL: {public_url}\n")

# thread = threading.Thread(target=uvicorn.run, args=(app,), kwargs={"host": "0.0.0.0", "port": 8000})
# thread.daemon = True
# thread.start()

# import time
# while True:
#     time.sleep(60)

In [4]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
huggingface_token = user_secrets.get_secret("huggingface token1")
login(token = huggingface_token)

In [5]:
import pandas as pd
df = pd.read_json(r"/kaggle/input/datasets/simonjayl/fintine-data-jsonl/fintune_data.jsonl", lines=True)

test_df = df.tail(750)

df.shape

(8483, 3)

In [6]:
import math
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def calculate_perplexity(model, tokenizer, texts):
    model.eval()
    total_loss = 0
    total_tokens = 0

    with torch.no_grad():
        for _, row in test_df.iterrows():

            prompt = f"[INST] Respond to this conversation as Simon would, match his texting style, tone and vocabulary exactly.\n\n {row['input']} [/INST]"
            full_text = f"{prompt} {row['output']}"

            prompt_inputs = tokenizer(prompt, return_tensors="pt")
            full_inputs = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=256).to(model.device)

            prompt_len = prompt_inputs["input_ids"].size(1)
            
            labels = full_inputs["input_ids"].clone()
            labels[:, :prompt_len] = -100
            outputs = model(**full_inputs, labels=labels)
            loss = outputs.loss
            shift_labels = labels[..., 1:]
            
            num_eval_tokens = (shift_labels != -100).sum().item()
            if num_eval_tokens > 0:
                total_loss += loss.item() * num_eval_tokens
                total_tokens += num_eval_tokens

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)
    return perplexity

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

base_name = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(base_name)

print(f"Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_name,
    quantization_config=bnb_config,
    device_map="auto"
)
base_ppl = calculate_perplexity(base_model, tokenizer, test_df)
print(f"Base model perplexity = {base_ppl:.2f}")

del base_model
gc.collect()
torch.cuda.empty_cache()

finetuned_name = "simonjayl/simon-chatbot"

print(f"Loading fine-tuned model...")
finetuned_model = AutoModelForCausalLM.from_pretrained(
    finetuned_name,
    quantization_config=bnb_config,
    device_map="auto"
)

finetuned_ppl = calculate_perplexity(finetuned_model, tokenizer, test_df)
print(f"Finetuned perplexity = {finetuned_ppl:.2f}")

improvement = ((base_ppl - finetuned_ppl) / base_ppl) * 100
print(f"\nImprovement: {improvement:.1f}% lower perplexity")

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/14.5G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/110 [00:00<?, ?B/s]

Finetuned perplexity = 24.19

Improvement: 99.6% lower perplexity


In [17]:
print(test_df['output'].isna().sum())
print(test_df['output'].apply(lambda x: len(str(x).strip())).describe())

0
count    750.000000
mean      37.586667
std       36.965359
min        1.000000
25%       15.000000
50%       26.000000
75%       49.000000
max      359.000000
Name: output, dtype: float64


In [10]:
first_10 = test_df.head(10)

for _, row in first_10.iterrows():
    prompt = f"[INST] Respond to this conversation as Simon would, match his texting style, tone and vocabulary exactly.\n\n {row['input']} [/INST]"
    full_text = f"{prompt} {row['output']}"

    prompt_inputs = tokenizer(prompt, return_tensors="pt")
    full_inputs = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=256)

    prompt_len = prompt_inputs["input_ids"].size(1)
    full_len = full_inputs["input_ids"].size(1)

    labels = full_inputs["input_ids"].clone()
    labels[:, :prompt_len] = -100
    shift_labels = labels[..., 1:]
    num_eval_tokens = (shift_labels != -100).sum().item()

    print(f"prompt_len: {prompt_len}, full_len: {full_len}, eval_tokens: {num_eval_tokens}")

prompt_len: 125, full_len: 132, eval_tokens: 7
prompt_len: 56, full_len: 60, eval_tokens: 4
prompt_len: 56, full_len: 63, eval_tokens: 7
prompt_len: 57, full_len: 61, eval_tokens: 4
prompt_len: 72, full_len: 79, eval_tokens: 7
prompt_len: 63, full_len: 68, eval_tokens: 5
prompt_len: 63, full_len: 69, eval_tokens: 6
prompt_len: 50, full_len: 58, eval_tokens: 8
prompt_len: 64, full_len: 69, eval_tokens: 5
prompt_len: 69, full_len: 85, eval_tokens: 16


In [14]:
total_tokens = 0

for _, row in test_df.iterrows():
    prompt = f"[INST] Respond to this conversation as Simon would, match his texting style, tone and vocabulary exactly.\n\n {row['input']} [/INST]"
    full_text = f"{prompt} {row['output']}"

    prompt_inputs = tokenizer(prompt, return_tensors="pt")
    full_inputs = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=256)

    prompt_len = prompt_inputs["input_ids"].size(1)
    full_len = full_inputs["input_ids"].size(1)

    labels = full_inputs["input_ids"].clone()
    labels[:, :prompt_len] = -100
    shift_labels = labels[..., 1:]
    row_tokens = (shift_labels != -100).sum().item()

    total_tokens += row_tokens

print(f"Total tokens: {total_tokens}, Length of test_df: {len(test_df)}")

Total tokens: 7738, Length of test_df: 750
